# Teks — POS Tagging / Pelabelan Urutan per Token (CRF)
**Kapan dipakai:** tiap kata diberi kelas kata (kata benda, kerja, sifat, ...). Pola yang sama
dipakai untuk chunking, tanda baca, atau label per token lain.

**Bedanya dengan NER:** hampir semua token punya tag penting, jadi metrik = akurasi token +
F1 macro per tag.

**Data demo:** POSP (IndoNLU, 26 tag).

In [1]:
import pandas as pd
import sklearn_crfsuite
from sklearn.metrics import accuracy_score, classification_report, f1_score

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/text/pos"   # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/posp_train.csv"
VALID_CSV = f"{DATA_FOLDER}/posp_valid.csv"
TEST_CSV = f"{DATA_FOLDER}/posp_test.csv"

SENTENCE_COL = "kalimat_id"
TOKEN_COL = "token"
TAG_COL = "tag"

## Baca data
Satu baris = satu token. `keep_default_na=False` supaya token seperti "NA" tidak jadi kosong.

In [3]:
train = pd.read_csv(TRAIN_CSV, keep_default_na=False)
valid = pd.read_csv(VALID_CSV, keep_default_na=False)
test = pd.read_csv(TEST_CSV, keep_default_na=False)
print("train:", train.shape, "| valid:", valid.shape, "| test:", test.shape)
print("test terurut per kalimat:", test[SENTENCE_COL].is_monotonic_increasing)
train.head(10)

train: (156409, 3) | valid: (19629, 3) | test: (19952, 3)
test terurut per kalimat: True


,kalimat_id,token,tag
0,0,kepala,B-NNO
1,0,dinas,B-VBP
2,0,tata,B-NNO
3,0,kota,B-NNO
4,0,manado,B-NNP
5,0,amos,B-NNP
6,0,kenda,B-NNP
7,0,menyatakan,B-VBT
8,0,tidak,B-NEG
9,0,tahu,B-VBI


## EDA: jumlah tiap tag dan panjang kalimat

In [4]:
print(train[TAG_COL].value_counts())
sentence_lengths = train.groupby(SENTENCE_COL).size()
print(sentence_lengths.describe())

tag
B-NNO    38316
B-NNP    28573
B-SYM    23421
B-PPO    12480
B-VBT     7629
B-NUM     6634
B-ADJ     4900
B-VBI     4515
B-VBP     4375
B-CCN     3805
B-ADV     3760
B-PRR     3371
B-ART     2889
B-CSN     2479
B-ADK     2168
B-PRN     2054
B-VBL     1176
B-NEG     1175
B-KUA     1109
B-VBE      513
B-PAR      353
B-PRI      263
B-PRK      245
B-UNS      141
B-$$$       52
B-INT       13
Name: count, dtype: int64
count    6720.000000
mean       23.275149
std        10.499649
min         1.000000
25%        16.000000
50%        22.000000
75%        29.000000
max       110.000000
dtype: float64


## Kelompokkan token per kalimat
Sudah ada file valid. Tidak ada? Pecah `kalimat_id` train dengan `GroupShuffleSplit`.

In [5]:
def to_sentences(table, col):
    sentences = []
    for sentence_id, group in table.groupby(SENTENCE_COL):
        sentences.append(list(group[col]))
    return sentences


train_tokens = to_sentences(train, TOKEN_COL)
train_tags = to_sentences(train, TAG_COL)
valid_tokens = to_sentences(valid, TOKEN_COL)
valid_tags = to_sentences(valid, TAG_COL)
test_tokens = to_sentences(test, TOKEN_COL)
print(len(train_tokens), "kalimat train,", len(valid_tokens), "valid,", len(test_tokens), "test")

6720 kalimat train, 840 valid, 840 test


## Feature per token
Tambah/hapus baris di fungsi ini untuk mencoba feature lain.

In [6]:
def token_features(sentence, i):
    word = sentence[i]
    features = {"bias": 1.0, "kata": word.lower(), "huruf_besar": word.istitle()}
    features["angka"] = word.isdigit()
    features["tanda_baca"] = not word.isalnum()
    features["akhiran2"] = word[-2:].lower()
    features["akhiran3"] = word[-3:].lower()
    features["awalan2"] = word[:2].lower()
    features["awalan3"] = word[:3].lower()
    if i > 0:
        features["kata_sebelum"] = sentence[i - 1].lower()
    else:
        features["awal_kalimat"] = True
    if i < len(sentence) - 1:
        features["kata_sesudah"] = sentence[i + 1].lower()
    else:
        features["akhir_kalimat"] = True
    return features


print(token_features(train_tokens[0], 0))

{'bias': 1.0, 'kata': 'kepala', 'huruf_besar': False, 'angka': False, 'tanda_baca': False, 'akhiran2': 'la', 'akhiran3': 'ala', 'awalan2': 'ke', 'awalan3': 'kep', 'awal_kalimat': True, 'kata_sesudah': 'dinas'}


## Ubah semua kalimat menjadi daftar feature

In [7]:
def all_features(sentences):
    result = []
    for sentence in sentences:
        sentence_features = []
        for i in range(len(sentence)):
            sentence_features.append(token_features(sentence, i))
        result.append(sentence_features)
    return result


X_train = all_features(train_tokens)
X_val = all_features(valid_tokens)
X_test = all_features(test_tokens)

## Model (CRF)
`c1`, `c2` = kekuatan regularisasi. Pilih SATU model: hapus `#` di baris yang ingin dipakai.

In [8]:
model = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100,
                             all_possible_transitions=True)
# model = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.01, max_iterations=200,
#                              all_possible_transitions=True)
# model = sklearn_crfsuite.CRF(algorithm="l2sgd", c2=0.1, max_iterations=200)
# model = sklearn_crfsuite.CRF(algorithm="ap", max_iterations=50)   # averaged perceptron

model.fit(X_train, train_tags)

,algorithm,'lbfgs'
,all_possible_transitions,True
,c1,0.1
,c2,0.1
,max_iterations,100
,min_freq,None
,all_possible_states,None
,num_memories,None
,epsilon,None
,period,None
,delta,None


## Evaluasi di valid (per token)
Tebakan per kalimat disambung menjadi satu daftar per token.

In [9]:
val_predictions = []
for sentence_tags in model.predict(X_val):
    val_predictions.extend(sentence_tags)
val_true = []
for sentence_tags in valid_tags:
    val_true.extend(sentence_tags)

print("akurasi token:", round(accuracy_score(val_true, val_predictions), 4))
print("F1 macro per tag:", round(f1_score(val_true, val_predictions, average="macro"), 4))
print(classification_report(val_true, val_predictions, digits=3, zero_division=0))

akurasi token: 0.9598
F1 macro per tag: 0.9637
              precision    recall  f1-score   support

       B-$$$      1.000     1.000     1.000        13
       B-ADJ      0.938     0.902     0.919       633
       B-ADK      1.000     0.987     0.993       300
       B-ADV      0.973     0.975     0.974       478
       B-ART      0.989     0.981     0.985       365
       B-CCN      0.998     0.998     0.998       484
       B-CSN      0.973     0.944     0.958       301
       B-INT      1.000     1.000     1.000         2
       B-KUA      0.962     0.984     0.973       129
       B-NEG      0.993     0.993     0.993       143
       B-NNO      0.930     0.953     0.941      4735
       B-NNP      0.937     0.919     0.928      3693
       B-NUM      0.995     0.966     0.980       791
       B-PAR      0.953     0.872     0.911        47
       B-PPO      0.981     0.990     0.986      1591
       B-PRI      1.000     0.964     0.982        28
       B-PRK      0.806     0.926 

## Latih ulang (train + valid) + submission
Urutan token sama dengan file test karena kalimat sudah terurut.

In [10]:
model.fit(X_train + X_val, train_tags + valid_tags)
predictions = []
for sentence_tags in model.predict(X_test):
    predictions.extend(sentence_tags)

submission = pd.DataFrame()
submission["kalimat_id"] = test["kalimat_id"]
submission["token"] = test["token"]
submission["tag"] = predictions
submission.to_csv("submission.csv", index=False)
submission.head()

,kalimat_id,token,tag
0,0,kuasa,B-NNO
1,0,hukum,B-NNO
2,0,teamster,B-NNP
3,0,berasal,B-VBI
4,0,dari,B-PPO


## Cek dengan kunci jawaban latihan
**HAPUS sel ini saat lomba**: kolom `tag` di file test hanya ada di data latihan.

In [11]:
print("akurasi token test:", round(accuracy_score(test[TAG_COL], predictions), 4))
print("F1 macro test:", round(f1_score(test[TAG_COL], predictions, average="macro"), 4))

akurasi token test: 0.9624
F1 macro test: 0.9519
